# Digital Banking Audit Tool
**Lesson 4 Individual Assignment**

**Author:** [Your Name] &nbsp;|&nbsp; **Course:** [Course Code and Name] &nbsp;|&nbsp; **Instructor:** [Instructor Name]

This notebook implements an audit tool for batch transaction files. It uses `filter()` and `map()` with lambda expressions to clean transactions and apply service fees, and a purely recursive function to total arbitrarily nested sub-transactions.

## Task 1: Cleaning Transactions and Applying Fees

* `clean_transactions()` uses `filter()` with a lambda to keep only positive amounts that are at least `min_threshold`.
* `apply_service_fee()` uses `map()` with a lambda to deduct a percentage fee from every amount.

Both functions use default arguments and return new lists without changing the input.

In [1]:
def clean_transactions(amounts, min_threshold=10.0):
    """Keep only positive transactions at or above min_threshold."""
    return list(filter(lambda amt: amt > 0 and amt >= min_threshold, amounts))


def apply_service_fee(amounts, fee_rate=0.02):
    """Deduct a percentage service fee from every transaction amount."""
    return list(map(lambda amt: round(amt * (1 - fee_rate), 2), amounts))

In [2]:
raw = [250, -40, 5, 0, 12.5, 100, 9.99]
print("Raw batch:        ", raw)

cleaned = clean_transactions(raw)
print("Cleaned (>=10.0): ", cleaned)

processed = apply_service_fee(cleaned)
print("After 2% fee:     ", processed)

Raw batch:         [250, -40, 5, 0, 12.5, 100, 9.99]
Cleaned (>=10.0):  [250, 12.5, 100]
After 2% fee:      [245.0, 12.25, 98.0]


## Task 2: Recursive Nested Total

`calculate_nested_total(data)` accepts a number or an arbitrarily nested list.

* **Base case 1:** an empty list returns `0.0`.
* **Base case 2:** a single `int` or `float` returns its own value.
* **Recursive step:** for a non-empty list, split into `head = data[0]` and `tail = data[1:]`, then return the total of the head plus the total of the tail.

No `for` or `while` loops and no global variables are used.

In [3]:
def calculate_nested_total(data):
    """
    Recursively total a number or an arbitrarily nested list of numbers.

    Base cases:
        [] -> 0.0
        int/float -> the value itself
    Recursive step:
        non-empty list -> total(head) + total(tail)
    """
    if isinstance(data, list):
        if data == []:
            return 0.0
        head = data[0]
        tail = data[1:]
        return calculate_nested_total(head) + calculate_nested_total(tail)
    if isinstance(data, (int, float)) and not isinstance(data, bool):
        return data
    raise TypeError(f"Unsupported type in batch: {type(data).__name__}")

In [4]:
nested = [100, [50, 25], [[10, 5], 40]]
print("Nested batch:     ", nested)
print("Nested total:     ", calculate_nested_total(nested))

print("Single value:     ", calculate_nested_total(75.5))
print("Empty list:       ", calculate_nested_total([]))
print("Deep nesting:     ", calculate_nested_total([1, [2, [3, [4, [5]]]]]))

Nested batch:      [100, [50, 25], [[10, 5], 40]]
Nested total:      230.0
Single value:      75.5
Empty list:        0.0
Deep nesting:      15.0


## Additional Edge-Case Tests

These checks cover empty input, boundary values, custom arguments, floats, invalid types, negative values, and the recursion depth limit.

In [5]:
cases = [
    ("clean_transactions([])", lambda: clean_transactions([])),
    ("clean_transactions([10, 10.0, 9.999])", lambda: clean_transactions([10, 10.0, 9.999])),
    ("clean_transactions([5, 20, 50], min_threshold=25)", lambda: clean_transactions([5, 20, 50], min_threshold=25)),
    ("clean_transactions([-5, 0, 3], min_threshold=-10)", lambda: clean_transactions([-5, 0, 3], min_threshold=-10)),
    ("apply_service_fee([])", lambda: apply_service_fee([])),
    ("apply_service_fee([200], fee_rate=0.1)", lambda: apply_service_fee([200], fee_rate=0.1)),
    ("apply_service_fee([19.99])", lambda: apply_service_fee([19.99])),
    ("calculate_nested_total([[], [[]], 5])", lambda: calculate_nested_total([[], [[]], 5])),
    ("calculate_nested_total([1.5, [2.25, [3]]])", lambda: calculate_nested_total([1.5, [2.25, [3]]])),
    ("calculate_nested_total([-10, [5]])", lambda: calculate_nested_total([-10, [5]])),
    ('calculate_nested_total(["50"])', lambda: calculate_nested_total(["50"])),
    ("calculate_nested_total([True])", lambda: calculate_nested_total([True])),
    ("calculate_nested_total(list(range(500)))", lambda: calculate_nested_total(list(range(500)))),
    ("calculate_nested_total(list(range(5000)))", lambda: calculate_nested_total(list(range(5000)))),
]
for label, call in cases:
    try:
        result = call()
    except (TypeError, RecursionError) as error:
        result = f"{type(error).__name__}: {error}"
    print(f"{label:<52} -> {result}")

clean_transactions([])                               -> []
clean_transactions([10, 10.0, 9.999])                -> [10, 10.0]
clean_transactions([5, 20, 50], min_threshold=25)    -> [50]
clean_transactions([-5, 0, 3], min_threshold=-10)    -> [3]
apply_service_fee([])                                -> []
apply_service_fee([200], fee_rate=0.1)               -> [180.0]
apply_service_fee([19.99])                           -> [19.59]
calculate_nested_total([[], [[]], 5])                -> 5.0
calculate_nested_total([1.5, [2.25, [3]]])           -> 6.75
calculate_nested_total([-10, [5]])                   -> -5.0
calculate_nested_total(["50"])                       -> TypeError: Unsupported type in batch: str
calculate_nested_total([True])                       -> TypeError: Unsupported type in batch: bool
calculate_nested_total(list(range(500)))             -> 124750.0
calculate_nested_total(list(range(5000)))            -> RecursionError: maximum recursion depth exceeded


## Trace and Limitations

`calculate_nested_total([100, [50, 25]])` = 100 + (50 + (25 + 0)) = **175**.

Python does not optimize tail calls and limits recursion depth to about 1,000 frames by default, so a very long flat list could raise a `RecursionError`. Slicing also copies the tail on every call. For production use, an iterative or stack-based approach would scale better.